# WX-1 자동 학습 (Colab One-Click)

**Run all** 누르면 끝. 손 안 대기.

1. GPU 확인
2. 라이브러리 설치
3. Drive 마운트
4. 리포지토리 클론
5. train_qlora.py 자동 패치
6. 데이터 생성 (기본 5000개)
7. 학습
8. LoRA 즉시 백업
9. 병합
10. GGUF f16
11. Q4_K_M 양자화
12. Drive 저장 + 다운로드

> 런타임 재시작 **필수 아님.** 모든 단계가 `!python ...` 서브프로세스로
> 실행되므로 새로 설치한 라이브러리를 깨끗한 프로세스에서 로드한다.
> (셀에서 직접 `import transformers` 하려면 그때만 재시작)

> 베이스는 **1.5B**. Colab 무료 T4 의 RAM 은 12.7GB 라 7B 병합(15GB) 은
> OOM 으로 죽는다. 7B 가 필요하면 Kaggle 노트북을 쓸 것.

In [ ]:
# === 설정 ===
GITHUB_USER = "ijeonghu522-stack"
GITHUB_REPO = "wuxicode"
TRAIN_MODEL = "wx-1-coder"

# 베이스 모델. 7B 는 병합 단계에서 RAM 18GB 필요 -> Colab 무료 T4 는 불가.
# 무료 런타임으로 돌리려면 1.5B 를 쓰세요.
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

MAX_SAMPLES = 5000
EPOCHS = 3
SEQ_LEN = 4096

print("WX-1 설정")
print(f"  repo    : {GITHUB_USER}/{GITHUB_REPO}")
print(f"  model   : {TRAIN_MODEL}")
print(f"  base    : {BASE_MODEL}")
print(f"  samples : {MAX_SAMPLES}")
print(f"  epochs  : {EPOCHS}  seq_len={SEQ_LEN}")


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q --upgrade "numpy>=2.0" "transformers>=4.56" "trl>=0.12" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" "pyyaml>=6.0"


In [ ]:
# 런타임 재시작: 필요하지 않습니다.
# 학습/병합/변환이 모두 `!python ...` 서브프로세스로 실행되므로
# 새로 설치한 numpy/transformers/trl 을 깨끗한 프로세스에서 로드합니다.
# (직접 셀에서 import 하려면 런타임 > 런타임 다시 시작)
print("=" * 56)
print("  런타임 재시작 불필요 — 이후 셀은 별도 프로세스에서 실행됩니다.")
print("=" * 56)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("[OK] Drive 마운트")


In [ ]:
!rm -rf /content/wuxicode
!git clone --depth 1 https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git /content/wuxicode
%cd /content/wuxicode
!ls training/


In [ ]:
# train_qlora.py 를 최신 TRL(SFTConfig) 스타일로 패치
!python training/fix_train_qlora.py


In [ ]:
# 데이터셋 생성 (RUN_ALL 에서 설정값 사용)
!python training/auto_build_datasets.py {MAX_SAMPLES}
!wc -l datasets/wx_coder.jsonl


In [ ]:
# config 를 노트북 설정값에 맞춰 조정 (들여쓰기 보존)
import re

path = "training/config/wx-1-coder.yaml"
s = open(path, encoding="utf-8").read()


def set_key(text, key, value):
    # 원래 들여쓰기를 그대로 유지해야 YAML 이 깨지지 않는다
    pat = r"^(\s*)" + re.escape(key) + r":.*$"
    return re.sub(pat, lambda m: f"{m.group(1)}{key}: {value}", text, flags=re.MULTILINE)


s = set_key(s, "base_model", BASE_MODEL)
s = set_key(s, "max_samples", MAX_SAMPLES)
s = set_key(s, "num_train_epochs", EPOCHS)
s = set_key(s, "max_seq_length", SEQ_LEN)

open(path, "w", encoding="utf-8").write(s)

import yaml

cfg = yaml.safe_load(s)
print("base_model  :", cfg["base_model"])
print("max_samples :", cfg["dataset"]["max_samples"])
print("epochs      :", cfg["training"]["num_train_epochs"])
print("max_seq_len :", cfg["training"]["max_seq_length"])


In [ ]:
!python training/train_qlora.py --config training/config/wx-1-coder.yaml


In [ ]:
# 학습 직후 즉시 백업 (Colab 세션 끊겨도 살아남음)
import shutil
from pathlib import Path

out = Path("output") / "{TRAIN_MODEL}-lora"
if not out.exists():
    print(f"[FAIL] {out} 없음. 위 학습 셀 로그를 확인하세요.")
else:
    zip_name = shutil.make_archive(str(out), "zip", out)
    shutil.copy(zip_name, "/content/drive/MyDrive/")
    print(f"[OK] LoRA 백업: /MyDrive/{Path(zip_name).name}")


In [ ]:
!python training/merge_adapter.py \
    --base {BASE_MODEL} \
    --adapter output/{TRAIN_MODEL}-lora \
    --output output/{TRAIN_MODEL}-merged \
    --dtype float16


In [ ]:
!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!pip install -q "protobuf>=5.29.0"


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py output/{TRAIN_MODEL}-merged \
    --outfile {TRAIN_MODEL}-f16.gguf \
    --outtype f16


In [ ]:
!cmake -B llama.cpp/build -S llama.cpp -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build llama.cpp/build --config Release -j4 --target llama-quantize


In [ ]:
!./llama.cpp/build/bin/llama-quantize \
    {TRAIN_MODEL}-f16.gguf \
    {TRAIN_MODEL}-q4_k_m.gguf \
    Q4_K_M


In [ ]:
import shutil
from pathlib import Path

gguf = Path("{TRAIN_MODEL}-q4_k_m.gguf")
if not gguf.exists():
    raise SystemExit(f"[FAIL] {gguf} 없음. 양자화 셀 로그를 확인하세요.")

shutil.copy(gguf, "/content/drive/MyDrive/")
size_gb = gguf.stat().st_size / (1024 ** 3)
print(f"[OK] Drive 저장: /MyDrive/{gguf.name}  ({size_gb:.2f} GB)")

from google.colab import files

files.download(str(gguf))


In [ ]:
print("=" * 50)
print("  완료")
print("  Drive : /MyDrive/{TRAIN_MODEL}-q4_k_m.gguf")
print("  LoRA  : /MyDrive/{TRAIN_MODEL}-lora.zip")
print("=" * 50)
